In [ ]:
import json
import zipfile

import httpx

In [ ]:
def get_token(url="http://localhost:8000", username="developer", password="123456"):
    keycloak_token = httpx.post(f"{url}/keycloak/realms/bluecore/protocol/openid-connect/token",
                      data={
                          "client_id": "bluecore_api",
                          "username": username,
                          "password": password,
                          "grant_type": "password",
                      })
    
    keycloak_token.raise_for_status()
    return keycloak_token.json().get('access_token')

In [ ]:
bc_headers = {
    "Authorization": f"Bearer {get_token()}"
}

with zipfile.ZipFile("sinopia_profiles.zip") as profiles_zip:
    files = profiles_zip.namelist()
    for i,profile in enumerate(files):
        print(f"Adding {profile} to Local Blue Core")
        with profiles_zip.open(profile) as profile_file:
            profile_contents = json.loads(profile_file.read())
        post_result = httpx.post("http://localhost:8000/api/resources",
                                 headers=bc_headers,
                                 follow_redirects=True,
                                 json={
                                     "data": json.dumps(profile_contents),
                                     "is_profile": True,
                                     "uri": f"http://localhost:8000/api/resources/{i+1}"
                                 })
        post_result.raise_for_status()